# Advanced Professional Baseline Notebook Structure

## ROGII - Wellbore Geology Prediction

---

# 1. Notebook Header

```python
# ============================================================
# ROGII - Wellbore Geology Prediction
# Advanced Baseline Pipeline
#
# Author: Md Ashraf
# IIT (ISM) Dhanbad
# ============================================================
```

---

In [3]:

from IPython.core import getipython
from IPython.core import getipython
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 200)

# =============================
# PATHS
# =============================

ROOT_DIR = Path("/kaggle/input/rogii-wellbore-geology-prediction")

TRAIN_DIR = ROOT_DIR / "train"
TEST_DIR = ROOT_DIR / "test"

SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# 4. Competition Overview
print("="*60)
print("ROGII - Wellbore Geology Prediction")
print("Target: Predict TVT along horizontal wells")
print("Metric: RMSE")
print("="*60)




ROGII - Wellbore Geology Prediction
Target: Predict TVT along horizontal wells
Metric: RMSE


In [2]:
pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 299.8 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:

# 5. Helper Functions

## RMSE Function

# ============================================================
# METRIC
# ============================================================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ============================================================
# MEMORY OPTIMIZATION
# ============================================================

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)

            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    return df


# 6. Data Loading

## Read Horizontal Wells


# ============================================================
# LOAD TRAIN DATA
# ============================================================

train_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))

train_data = []

for file in tqdm(train_files):

    well_name = file.stem.split("__")[0]

    df = pd.read_csv(file)

    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))

    train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)

print(train_df.shape)
train_df.head()

## Load Test Data

# ============================================================
# LOAD TEST DATA
# ============================================================

test_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))

test_data = []

for file in tqdm(test_files):

    well_name = file.stem.split("__")[0]

    df = pd.read_csv(file)

    df["WELL"] = well_name
    df["ROW_IDX"] = np.arange(len(df))

    test_data.append(df)


test_df = pd.concat(test_data, ignore_index=True)

print(test_df.shape)
test_df.head()

0it [00:00, ?it/s]

ValueError: No objects to concatenate

In [ ]:

```

---

# 7. Exploratory Data Analysis (EDA)

## Missing Values

```python
missing = train_df.isnull().mean().sort_values(ascending=False)
missing[missing > 0]
```

---

## Target Distribution

```python
fig = px.histogram(
    train_df,
    x="TVT",
    nbins=100,
    title="TVT Distribution"
)
fig.show()
```

---

## Well-wise Visualization

```python
sample_well = train_df["WELL"].unique()[0]

well_df = train_df[train_df["WELL"] == sample_well]

fig = px.line(
    well_df,
    x="MD",
    y=["GR", "TVT"],
    title=f"Well Analysis: {sample_well}"
)

fig.show()
```

---

# 8. Geological Feature Engineering

## Surface Relative Features

```python
# ============================================================
# GEOLOGICAL RELATIVE POSITION FEATURES
# ============================================================

surface_cols = [
    "ANCC", "ASTNU", "ASTNL",
    "EGFDU", "EGFDL", "BUDA"
]

for col in surface_cols:
    train_df[f"Z_minus_{col}"] = train_df["Z"] - train_df[col]
    test_df[f"Z_minus_{col}"] = test_df["Z"] - test_df[col]
```

---

## Rolling Window Features

```python
# ============================================================
# ROLLING FEATURES
# ============================================================

windows = [5, 10, 20, 50]

for window in windows:

    train_df[f"GR_roll_mean_{window}"] = (
        train_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    train_df[f"GR_roll_std_{window}"] = (
        train_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).std())
    )

    test_df[f"GR_roll_mean_{window}"] = (
        test_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )

    test_df[f"GR_roll_std_{window}"] = (
        test_df.groupby("WELL")["GR"]
        .transform(lambda x: x.rolling(window, min_periods=1).std())
    )
```

---

## Lag Features

```python
# ============================================================
# LAG FEATURES
# ============================================================

lags = [1, 2, 5, 10]

for lag in lags:

    train_df[f"GR_lag_{lag}"] = (
        train_df.groupby("WELL")["GR"].shift(lag)
    )

    test_df[f"GR_lag_{lag}"] = (
        test_df.groupby("WELL")["GR"].shift(lag)
    )
```

---

## Gradient Features

```python
# ============================================================
# GRADIENT FEATURES
# ============================================================

train_df["GR_gradient"] = (
    train_df.groupby("WELL")["GR"].diff()
)

test_df["GR_gradient"] = (
    test_df.groupby("WELL")["GR"].diff()
)
```

---

## Trajectory Features

```python
# ============================================================
# TRAJECTORY FEATURES
# ============================================================

train_df["dX"] = train_df.groupby("WELL")["X"].diff()
train_df["dY"] = train_df.groupby("WELL")["Y"].diff()
train_df["dZ"] = train_df.groupby("WELL")["Z"].diff()

train_df["trajectory_distance"] = np.sqrt(
    train_df["dX"]**2 +
    train_df["dY"]**2 +
    train_df["dZ"]**2
)


test_df["dX"] = test_df.groupby("WELL")["X"].diff()
test_df["dY"] = test_df.groupby("WELL")["Y"].diff()
test_df["dZ"] = test_df.groupby("WELL")["Z"].diff()


test_df["trajectory_distance"] = np.sqrt(
    test_df["dX"]**2 +
    test_df["dY"]**2 +
    test_df["dZ"]**2
)
```

---

# 9. Feature Selection

```python
# ============================================================
# FEATURE LIST
# ============================================================

exclude_cols = [
    "TVT",
    "WELL",
]

features = [
    col for col in train_df.columns
    if col not in exclude_cols
]

print(f"Total Features: {len(features)}")
```

---

# 10. Prepare Training Data

```python
# ============================================================
# TARGET
# ============================================================

train_mask = train_df["TVT"].notnull()

X = train_df.loc[train_mask, features]
y = train_df.loc[train_mask, "TVT"]

groups = train_df.loc[train_mask, "WELL"]

X_test = test_df[features]
```

---

# 11. LightGBM Baseline Model

## Model Parameters

```python
# ============================================================
# LIGHTGBM PARAMETERS
# ============================================================

params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "random_state": 42,
    "n_estimators": 5000,
    "verbosity": -1,
}
```

---

# 12. Cross Validation

```python
# ============================================================
# GROUP K-FOLD VALIDATION
# ============================================================

folds = GroupKFold(n_splits=5)


oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

scores = []

for fold, (train_idx, valid_idx) in enumerate(
    folds.split(X, y, groups)
):

    print(f"\n========== Fold {fold+1} ==========")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    valid_pred = model.predict(X_valid)

    oof[valid_idx] = valid_pred

    fold_rmse = rmse(y_valid, valid_pred)

    scores.append(fold_rmse)

    print(f"Fold RMSE: {fold_rmse:.5f}")

    preds += model.predict(X_test) / folds.n_splits

print("\n==============================")
print(f"Mean RMSE: {np.mean(scores):.5f}")
print("==============================")
```

---

# 13. Feature Importance

```python
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

fig = px.bar(
    importance_df.head(30),
    x="importance",
    y="feature",
    orientation="h",
    title="Top Feature Importance"
)

fig.show()
```

---

# 14. Submission Generation

```python
# ============================================================
# SUBMISSION FILE
# ============================================================

submission_rows = []

for well_name, df in test_df.groupby("WELL"):

    for idx, pred in zip(df.index, preds[df.index]):

        submission_rows.append({
            "id": f"{well_name}_{idx}",
            "tvt": pred
        })

submission = pd.DataFrame(submission_rows)

submission.to_csv("submission.csv", index=False)

submission.head()
```

---

# 15. Future Improvements

```python
# ============================================================
# FUTURE IMPROVEMENTS
# ============================================================

# TODO:
# 1. Typewell correlation features
# 2. Dynamic Time Warping (DTW)
# 3. Electrofacies clustering
# 4. LSTM sequence models
# 5. Transformer models
# 6. Ensemble learning
# 7. Physics-guided ML
# 8. Uncertainty quantification
```

---

# 16. Key Learnings Section

```python
print("="*60)
print("KEY INSIGHTS")
print("="*60)
print("1. Geological continuity is critical")
print("2. Sequential context matters")
print("3. Leakage prevention is essential")
print("4. Feature engineering dominates")
print("5. GroupKFold validation is mandatory")
print("="*60)
```

---

# 17. Advanced Next Steps

## Planned Research Directions

* Sequence-to-sequence geology prediction
* Transformer-based trajectory modeling
* Typewell alignment and correlation
* Real-time geosteering simulation
* Geological uncertainty estimation
* Hybrid physics + machine learning systems

---

# 18. Final Notes

This notebook establishes a strong and leakage-free baseline for the ROGII Wellbore Geology Prediction competition using:

* geological feature engineering
* trajectory analytics
* rolling statistical features
* GroupKFold validation
* LightGBM regression

The next stage is to incorporate:

* advanced sequence learning
* typewell correlation
* geological context modeling
* ensemble learning
* physics-guided AI
